# Parte 2 — Diagnósticos de regresión con datos horarios del SIMAT

Segundo notebook del curso con el mismo CSV de contaminantes. Aquí trabajamos con **todas las estaciones y todos los contaminantes** (CO, NO, NO2, NOX, O3, PM10, PM2.5, PMCO, SO2):

a) tabla ancha estación-hora × contaminante · b) covarianza y correlación de CO con NOX · c) eigenvalores de la matriz de correlación con y sin faltantes · d) regresión de PM2.5 contra los demás contaminantes (R², VIF) y luego solo con gases · e) matriz sombrero (H) para una estación y una semana, y leverage de todos sin construir H completa · f) distancia de Cook y reajuste sin los 10 puntos más influyentes.

**Fuente de datos:** https://aire.cdmx.gob.mx/descargas/Opendata/anuales_horarios/contaminantes_2026.csv

## 1. Librerías y carga de datos

In [1]:
import gdown
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

In [2]:
file_id = "1PcIYm8cIDgHolqvLkq15c_IgGp5y-WZc"
url = f"https://drive.google.com/uc?id={file_id}"
gdown.download(url, "contaminantes_2026.csv", quiet=False)

ESTACION = "MER"  # estacion usada en el inciso e); cambia aqui si quieres otra

# El archivo trae varias lineas de metadatos antes del encabezado real;
# buscamos la primera linea con forma de fila de datos (varias comas) y empezamos ahi.
with open("contaminantes_2026.csv", "r", encoding="utf-8", errors="replace") as f:
    lineas = f.readlines()
header_idx = next(i for i, l in enumerate(lineas) if l.count(",") >= 4)

df_raw = pd.read_csv("contaminantes_2026.csv", skiprows=header_idx)
print(df_raw.shape)
df_raw.head()

Downloading...
From: https://drive.google.com/uc?id=1PcIYm8cIDgHolqvLkq15c_IgGp5y-WZc
To: /content/contaminantes_2026.csv
100%|██████████| 17.6M/17.6M [00:00<00:00, 84.9MB/s]


(433296, 5)


,date,id_station,id_parameter,valor,unit
0,2026-01-01 00:00:00,ACO,CO,NaN,NaN
1,2026-01-01 00:00:00,ACO,NO,NaN,NaN
2,2026-01-01 00:00:00,ACO,NO2,NaN,NaN
3,2026-01-01 00:00:00,ACO,NOX,NaN,NaN
4,2026-01-01 00:00:00,ACO,O3,NaN,NaN


## 2. Revisar columnas

Igual que en el notebook anterior: el archivo viene en formato "largo" (`date`, `id_station`, `id_parameter`, `value`). Si tus nombres de columna son distintos, ajusta las 4 variables de la siguiente celda.

In [3]:
print(df_raw.columns.tolist())

COL_FECHA = "date"
COL_ESTACION = "id_station"
COL_PARAMETRO = "id_parameter"
COL_VALOR = "value"

df_raw[COL_FECHA] = pd.to_datetime(df_raw[COL_FECHA])
print(df_raw[COL_PARAMETRO].unique())

['date', 'id_station', 'id_parameter', 'valor', 'unit']
['CO' 'NO' 'NO2' 'NOX' 'O3' 'PM10' 'PM2.5' 'PMCO' 'SO2']


## a) Tabla con un renglón por estación-hora y una columna por contaminante

In [5]:
df_wide_all = (
    df_raw
    .pivot_table(index=[COL_FECHA, COL_ESTACION], columns=COL_PARAMETRO, values="valor", aggfunc="mean")
    .reset_index()
    .rename(columns={COL_FECHA: "date", COL_ESTACION: "id_station"})
    .sort_values(["date", "id_station"])
    .reset_index(drop=True)
)

contaminantes = [c for c in df_wide_all.columns if c not in ("date", "id_station")]
print(df_wide_all.shape, contaminantes)
df_wide_all.head()

(33982, 11) ['CO', 'NO', 'NO2', 'NOX', 'O3', 'PM10', 'PM2.5', 'PMCO', 'SO2']


id_parameter,date,id_station,CO,NO,NO2,NOX,O3,PM10,PM2.5,PMCO,SO2
0,2026-01-01,AJM,0.84,2.0,26.0,28.0,19.0,67.0,53.0,14.0,28.0
1,2026-01-01,ATI,0.80,2.0,21.0,23.0,18.0,90.0,NaN,NaN,22.0
2,2026-01-01,BJU,1.22,NaN,NaN,NaN,17.0,62.0,41.0,21.0,NaN
3,2026-01-01,CAM,NaN,7.0,44.0,52.0,4.0,57.0,49.0,8.0,25.0
4,2026-01-01,CCA,1.27,1.0,32.0,33.0,14.0,NaN,NaN,NaN,3.0


## b) Covarianza y correlación de CO con NOX. ¿Son distintas?

In [6]:
cov_co_nox = df_wide_all[["CO", "NOX"]].cov().loc["CO", "NOX"]
corr_co_nox = df_wide_all[["CO", "NOX"]].corr().loc["CO", "NOX"]

print(f"Covarianza(CO, NOX): {cov_co_nox:.4f}")
print(f"Correlación(CO, NOX): {corr_co_nox:.4f}")

Covarianza(CO, NOX): 22.3069
Correlación(CO, NOX): 0.8568


La covarianza y la correlación miden la misma relación lineal, pero no son lo mismo: la covarianza queda en las unidades de CO × NOX (por eso es difícil de interpretar sola), mientras que la correlación es la covarianza ya dividida entre las desviaciones estándar de cada variable, así que siempre queda entre -1 y 1 y sí se puede comparar entre pares de contaminantes distintos. El signo es el mismo en ambas; lo que cambia es la escala.

## c) Eigenvalores de la matriz de correlación, con y sin faltantes

In [7]:
# Con faltantes: correlación por pares (pandas ignora NaNs de dos en dos — .corr() por default)
corr_con_nan = df_wide_all[contaminantes].corr()
eigvals_con_nan = np.sort(np.linalg.eigvalsh(corr_con_nan.values))[::-1]

# Sin faltantes: solo las filas completas (mismos casos para todas las correlaciones)
df_completo = df_wide_all[contaminantes].dropna()
corr_sin_nan = df_completo.corr()
eigvals_sin_nan = np.sort(np.linalg.eigvalsh(corr_sin_nan.values))[::-1]

print("Eigenvalores con faltantes (correlación por pares):")
print(eigvals_con_nan)
print("\nEigenvalores sin faltantes (casos completos):")
print(eigvals_sin_nan)

Eigenvalores con faltantes (correlación por pares):
[ 4.14469495e+00  1.92135373e+00  1.03013140e+00  7.49362387e-01
  5.35746513e-01  4.71859906e-01  1.55245461e-01  2.90494944e-04
 -8.68484386e-03]

Eigenvalores sin faltantes (casos completos):
[4.18802141e+00 1.93994669e+00 1.04940566e+00 8.06154607e-01
 5.19993723e-01 3.71747078e-01 1.24379289e-01 2.82268611e-04
 6.92792696e-05]


Cuando la correlación se calcula "por pares" (cada celda con los datos disponibles para ese par de contaminantes, sin exigir que las demás columnas también estén completas), la matriz resultante ya no viene necesariamente de un solo conjunto de datos consistente, así que puede dejar de ser semidefinida positiva: si ves un eigenvalor negativo (o muy cercano a cero por errores de redondeo) ahí está la señal. Con los casos completos la matriz de correlación sí está garantizada semidefinida positiva, por eso todos sus eigenvalores deben salir ≥ 0.

## d) PM2.5 contra los demás contaminantes: R² y VIF. Luego solo con gases

In [8]:
predictores_todos = [c for c in contaminantes if c != "PM2.5"]
df_reg_todos = df_wide_all[contaminantes].dropna()

X_todos = sm.add_constant(df_reg_todos[predictores_todos])
y_todos = df_reg_todos["PM2.5"]
modelo_todos = sm.OLS(y_todos, X_todos).fit()

print("R² (PM2.5 contra todos los demás contaminantes):", modelo_todos.rsquared)

vif_todos = pd.DataFrame({
    "variable": X_todos.columns,
    "VIF": [variance_inflation_factor(X_todos.values, i) for i in range(X_todos.shape[1])],
})
vif_todos

R² (PM2.5 contra todos los demás contaminantes): 0.9987783393392791


,variable,VIF
0,const,1.000000
1,CO,5.541771
2,NO,5023.552603
3,NO2,1041.724208
4,NOX,8372.816244
5,O3,1.587416
6,PM10,3.814067
7,PMCO,3.074750
8,SO2,1.090556


In [9]:
# Solo con gases (sin material particulado: fuera PM10 y PMCO)
gases = [c for c in ["CO", "NO", "NO2", "NOX", "O3", "SO2"] if c in contaminantes]
df_reg_gases = df_wide_all[gases + ["PM2.5"]].dropna().reset_index(drop=True)

X_gases = sm.add_constant(df_reg_gases[gases])
y_gases = df_reg_gases["PM2.5"]
modelo_gases = sm.OLS(y_gases, X_gases).fit()

print("R² (PM2.5 contra solo gases):", modelo_gases.rsquared)

vif_gases = pd.DataFrame({
    "variable": X_gases.columns,
    "VIF": [variance_inflation_factor(X_gases.values, i) for i in range(X_gases.shape[1])],
})
vif_gases

R² (PM2.5 contra solo gases): 0.26652842247298714


,variable,VIF
0,const,1.000000
1,CO,4.216003
2,NO,5172.823642
3,NO2,935.342171
4,NOX,8181.523804
5,O3,1.334014
6,SO2,1.032705


## e) Matriz sombrero (H): una estación y una semana

Trabajamos con el modelo de gases del inciso anterior, pero solo con la primera semana de datos de `ESTACION`, para poder construir H completa sin que sea demasiado grande.

In [10]:
df_estacion = df_wide_all[df_wide_all["id_station"] == ESTACION].sort_values("date")
semana_fin = df_estacion["date"].min() + pd.Timedelta(days=7)
df_semana = df_estacion[df_estacion["date"] < semana_fin][gases + ["PM2.5"]].dropna()

X_semana = sm.add_constant(df_semana[gases]).values
H = X_semana @ np.linalg.inv(X_semana.T @ X_semana) @ X_semana.T

p = X_semana.shape[1]
print("¿H² ≈ H?:", np.allclose(H @ H, H))
print(f"traza(H) = {np.trace(H):.4f}   p = {p}")

¿H² ≈ H?: True
traza(H) = 7.0000   p = 7


Ahora el leverage de **todas** las observaciones (todas las estaciones, no solo una semana), sin construir la H completa — con esa cantidad de filas, H sería demasiado grande para tenerla en memoria. En vez de eso usamos directamente la fórmula por fila, $h_{ii} = x_i^\top (X^\top X)^{-1} x_i$, que solo requiere matrices de tamaño n×p y p×p.

In [11]:
X_full = X_gases.values
XtX_inv = np.linalg.inv(X_full.T @ X_full)
leverage = np.einsum("ij,jk,ik->i", X_full, XtX_inv, X_full)

# statsmodels calcula esto internamente con el mismo tipo de atajo (sin construir H completa);
# lo usamos aquí solo para confirmar que el cálculo manual coincide.
leverage_sm = modelo_gases.get_influence().hat_matrix_diag
print("leverage manual == statsmodels:", np.allclose(leverage, leverage_sm))

df_reg_gases["leverage"] = leverage
df_reg_gases[["leverage"]].describe()

leverage manual == statsmodels: True


id_parameter,leverage
count,9247.000000
mean,0.000757
std,0.001065
min,0.000122
25%,0.000327
50%,0.000524
75%,0.000819
max,0.031583


## f) Distancia de Cook: ¿los 10 más influyentes son los de más leverage?

In [12]:
cooks_d = modelo_gases.get_influence().cooks_distance[0]
df_reg_gases["cooks_d"] = cooks_d

top10_cooks = df_reg_gases.sort_values("cooks_d", ascending=False).head(10)
top10_leverage = df_reg_gases.sort_values("leverage", ascending=False).head(10)

overlap = set(top10_cooks.index) & set(top10_leverage.index)
print(f"Coinciden {len(overlap)} de los 10 puntos entre distancia de Cook y leverage")
top10_cooks[gases + ["PM2.5", "leverage", "cooks_d"]]

Coinciden 0 de los 10 puntos entre distancia de Cook y leverage


id_parameter,CO,NO,NO2,NOX,O3,SO2,PM2.5,leverage,cooks_d
55,2.82,87.0,31.0,119.0,5.0,2.0,323.0,0.005264,0.266157
63,3.11,100.0,44.0,144.0,6.0,4.0,296.0,0.004793,0.187243
18,2.62,68.0,43.0,112.0,4.0,5.0,324.0,0.003618,0.181215
31,2.33,51.0,36.0,87.0,5.0,3.0,290.0,0.002973,0.119758
46,3.26,103.0,33.0,136.0,8.0,3.0,182.0,0.006859,0.078146
72,2.15,30.0,59.0,89.0,19.0,7.0,305.0,0.001625,0.069071
11,2.72,57.0,45.0,102.0,6.0,7.0,201.0,0.003896,0.058642
38,2.18,56.0,33.0,89.0,5.0,2.0,203.0,0.002467,0.043851
53,1.76,34.0,26.0,60.0,8.0,3.0,202.0,0.001900,0.035575
48,1.79,53.0,35.0,87.0,3.0,3.0,201.0,0.001610,0.029635


In [13]:
# Reajustar el modelo sin los 10 puntos de mayor distancia de Cook
df_sin_influyentes = df_reg_gases.drop(index=top10_cooks.index)

X_sin = sm.add_constant(df_sin_influyentes[gases])
y_sin = df_sin_influyentes["PM2.5"]
modelo_sin_influyentes = sm.OLS(y_sin, X_sin).fit()

print("R² original:", modelo_gases.rsquared)
print("R² sin los 10 más influyentes:", modelo_sin_influyentes.rsquared)
print("\nCoeficientes originales:\n", modelo_gases.params)
print("\nCoeficientes sin los 10 más influyentes:\n", modelo_sin_influyentes.params)

R² original: 0.26652842247298714
R² sin los 10 más influyentes: 0.30309607957485607

Coeficientes originales:
 const     1.236289
CO       13.297976
NO       -0.507279
NO2      -0.198775
NOX       0.455374
O3        0.174096
SO2       0.212415
dtype: float64

Coeficientes sin los 10 más influyentes:
 const    1.337147
CO       8.537917
NO      -0.379543
NO2     -0.011858
NOX      0.354358
O3       0.183802
SO2      0.211673
dtype: float64
